# 8. Significance tests

The Section 8 battery (Pesaran-Timmermann, Diebold-Mariano, Jobson-Korkie/Memmel, Anatolyev-Gerko and the Deflated Sharpe ratio) on the held-out test period. It includes the backtester (same cells as notebook 5), then prints the Table 7 and Table 8 LaTeX and saves `fig_significance` (PNG and PDF). The full run does ~150 backtests, so give it a few minutes.

In [ ]:
# Setup - load the two prediction universals
# df_train = the window where hi/K are CHOSEN; df_validation = the held-out window that is CHECKED.
# Default = the 2021-22 (val) / 2023+ (test) split. To run the 2008 crisis experiment, generate the
# crisis parquets with the three Crisis notebooks, then swap the two PATHs + the WIN dict below.
import numpy as np, pandas as pd
DATA_DIR = "Data"

TRAIN_PATH = f"{DATA_DIR}/Predictions_45_val/all_predictions_val.parquet"   # 2021-2022  -> choose hi/K
VALID_PATH = f"{DATA_DIR}/Predictions_45/all_predictions.parquet"           # 2023+      -> held out
# crisis split:
# TRAIN_PATH = f"{DATA_DIR}/Predictions_crisis_val/all_predictions.parquet"   # 2006-2007  (Validation_Crisis)
# VALID_PATH = f"{DATA_DIR}/Predictions_crisis/all_predictions.parquet"       # 2008-2012  (Train_Crisis)

WIN = {"train": ("2021-01-01", "2022-12-31"), "valid": ("2023-01-01", None)}   # default windows
# crisis:  WIN = {"train": ("2006-01-01", "2007-12-31"), "valid": ("2008-01-01", "2012-12-31")}
RESULTS_DIR = f"{DATA_DIR}/Backtest_results"   # the Crisis analysis tab overrides this in its Setup

def read_preds(path):
    d = pd.read_parquet(path)
    d["train_start"] = d["train_start"].astype(str).str[:10]
    d["x_days"] = d["x_days"].astype(int); d["y_up"] = d["y_up"].astype(int)
    return d.reset_index(drop=True)

df_train = read_preds(TRAIN_PATH)
df_validation = read_preds(VALID_PATH)
def win(df): return WIN["train"] if df is df_train else WIN["valid"]

print("df_train", df_train.shape, "| df_validation", df_validation.shape)
print("algos", sorted(df_train.algo.unique()), "| tiers", sorted(df_train.tier.unique()),
      "| starts", sorted(df_train.train_start.unique()))
df_train.head()

In [ ]:
# Features + anchors + a per-slice viewer
# The parquets carry no dates, so predictions map to the first N period anchors rebuilt from the features table:
# daily = every trading day, weekly/monthly = first trading day of each period.
features = pd.concat([pd.read_parquet(f"{DATA_DIR}/features_1.parquet"),
                      pd.read_parquet(f"{DATA_DIR}/features_2.parquet")], ignore_index=True)

def anchors(ticker, scale):
    ft = features[features.ticker == ticker].sort_values("Date")
    if scale == "daily": return ft.reset_index(drop=True)
    key = ft["Date"].dt.to_period("W" if scale == "weekly" else "M")
    return ft.groupby(key, as_index=False).first().sort_values("Date").reset_index(drop=True)

def show(df, ticker, train_start="2000-01-01", x_days=30, algo="lstm", scale="weekly", fusion=None):
    if fusion is None: fusion = "single" if scale == "daily" else "together"
    m = ((df.ticker==ticker)&(df.algo==algo)&(df.tier=="gmm")&(df.scale==scale)&(df.fusion==fusion)
         &(df.train_start==train_start)&(df.x_days==x_days))
    sel = df[m].reset_index(drop=True)
    start, end = win(df)
    a = anchors(ticker, scale); a = a[a.Date >= pd.Timestamp(start)]
    if end: a = a[a.Date <= pd.Timestamp(end)]
    a = a.reset_index(drop=True); n = min(len(sel), len(a)); out = sel.iloc[:n].copy()
    for c in ["Date","Open","High","Low","Close"]: out[c] = a[c].to_numpy()[:n]
    keep = ["Date","ticker","prob_up","y_up"] + (["regime"] if "regime" in out.columns else [])
    return out[keep]

show(df_train, "AAPL", scale="weekly", fusion="split", train_start="2015-01-01", x_days=100)

In [ ]:
# Capital / fee, the 1/N benchmark, and a max-drawdown helper
# 1/N = buy an equal dollar of all 40 on the first trading day and hold to the last (scale-independent,
# pays the fee once). Both the strategy and 1/N start at $100k, so end value / return% / MDD compare directly.
CAPITAL, FEE = 100000.0, 0.0005     # 5 bps per trade (IBKR-realistic); FEE is a global read by every function

def portfolio_1overN(df):
    start, end = win(df); op, cl = {}, {}
    for tk in sorted(df.ticker.unique()):
        d = features[features.ticker==tk].sort_values("Date"); d = d[d.Date >= pd.Timestamp(start)]
        if end: d = d[d.Date <= pd.Timestamp(end)]
        op[tk] = pd.Series(d["Open"].to_numpy(),  index=d["Date"].to_numpy())
        cl[tk] = pd.Series(d["Close"].to_numpy(), index=d["Date"].to_numpy())
    op = pd.DataFrame(op).sort_index(); cl = pd.DataFrame(cl).sort_index()
    shares = (CAPITAL/len(op.columns)) / op.iloc[0]
    return cl.mul(shares, axis=1).sum(axis=1) - FEE*CAPITAL

def mdd_of(v):
    v = np.asarray(v, float); return round(float((1 - v/np.maximum.accumulate(v)).max()*100), 2)

print(f"1/N: train {portfolio_1overN(df_train).iloc[-1]:,.0f} | validation {portfolio_1overN(df_validation).iloc[-1]:,.0f}")

In [ ]:
# The gate fit on TRAIN - per-ticker buy threshold hi, and the tradable regimes
# fit_hi sweeps a buy/sell band per ticker and keeps the hi of the best acted-accuracy band (coverage >= min_cov).
# build_trad_cfg marks a regime tradable if its train directional accuracy >= min_acc with >= min_obs obs.
# Both are always fit on df_train and then applied to whichever window is backtested (kept out-of-sample).
def fit_hi(df, cfg, by="acc_acted", min_cov=0.3,
           his=np.round(np.arange(0.50,0.86,0.02),3), los=np.round(np.arange(0.15,0.51,0.02),3)):
    m = np.logical_and.reduce([df[c]==v for c,v in cfg.items() if c in df.columns]); sub=df[m]; hi_map={}
    for tk in sorted(sub.ticker.unique()):
        s=sub[sub.ticker==tk]; y=s["y_up"].to_numpy(); p=s["prob_up"].to_numpy(); best=None
        for hi in his:
            for lo in los:
                if lo>=hi: continue
                buy=p>=hi; sell=p<=lo; acted=buy|sell; cov=acted.mean()
                if cov<min_cov or not acted.any(): continue
                acc=((buy&(y==1))|(sell&(y==0)))[acted].mean(); edge=cov*(acc-0.5)
                score=acc if by=="acc_acted" else edge
                if best is None or (score,edge)>best[:2]: best=(score,edge,hi)
        if best is not None: hi_map[tk]=best[2]
    return hi_map

def build_trad_cfg(df, cfg, min_acc=0.50, min_obs=20):
    m = np.logical_and.reduce([df[c]==v for c,v in cfg.items() if c in df.columns]); sub=df[m]; trad={}
    for tk in sorted(sub.ticker.unique()):
        s=sub[sub.ticker==tk]
        if "regime" not in s.columns or s["regime"].isna().all(): trad[tk]=set(); continue
        d=s.dropna(subset=["regime"]).copy(); d["hit"]=((d.prob_up>0.5)==(d.y_up==1))
        g=d.groupby(d.regime).agg(n=("hit","size"), acc=("hit","mean"))
        trad[tk]=set(g.index[(g.acc>=min_acc)&(g.n>=min_obs)])
    return trad

In [ ]:
# Per-ticker signals + daily bars for one config on one window
# elig = prob >= its own hi (and, with GMM on, regime is tradable). Signals sit on the anchor dates;
# daily OHLC bars run over the window for the intraday TP/SL check.
def tpsl_data(df, cfg, hi_map, trad=None, use_regime=False):
    start, end = win(df)
    m = np.logical_and.reduce([df[c]==v for c,v in cfg.items() if c in df.columns]); sub=df[m]
    tickers=sorted(sub.ticker.unique()); sig, daily = {}, {}
    for tk in tickers:
        s=sub[sub.ticker==tk].reset_index(drop=True)
        aw=anchors(tk,cfg["scale"]); aw=aw[aw.Date>=pd.Timestamp(start)]
        if end: aw=aw[aw.Date<=pd.Timestamp(end)]
        aw=aw.reset_index(drop=True); n=min(len(s),len(aw))
        prob=s["prob_up"].to_numpy()[:n]; reg=s["regime"].to_numpy()[:n] if "regime" in s.columns else np.full(n,np.nan)
        elig=prob>=hi_map.get(tk,np.inf)
        if use_regime and trad is not None:
            tr=trad.get(tk)
            if tr is not None: elig=elig & np.isin(reg,list(tr))
        sig[tk]=pd.DataFrame({"prob":prob,"regime":reg,"elig":elig}, index=pd.to_datetime(aw["Date"].to_numpy()[:n]))
        d=features[features.ticker==tk].sort_values("Date"); d=d[d.Date>=pd.Timestamp(start)]
        if end: d=d[d.Date<=pd.Timestamp(end)]
        daily[tk]=d.set_index("Date")[["Open","High","Low","Close"]]
    return tickers, sig, daily

In [ ]:
# Pack one config into numpy arrays for the fast engine (K-independent, so prep once and sweep K)
# NEXT-BAR EXECUTION: each anchor's signal is placed on the FOLLOWING trading day (di[d]+1), so the engine
# trades at the open of the day AFTER the signal is known - the prediction uses the anchor day's close, so
# entering at that day's open would be a look-ahead. The last anchor is dropped (no next bar to trade it).
def tpsl_prep(df, cfg, hi_map, trad=None, use_regime=False):
    tickers, sig, daily = tpsl_data(df, cfg, hi_map, trad, use_regime)
    days=pd.DatetimeIndex(sorted(set().union(*[set(daily[tk].index) for tk in tickers]))); di={d:i for i,d in enumerate(days)}; N=len(days)
    O={};H={};L={};C={}
    for tk in tickers:
        d=daily[tk].reindex(days); O[tk]=d.Open.to_numpy();H[tk]=d.High.to_numpy();L[tk]=d.Low.to_numpy();C[tk]=d.Close.to_numpy()
    rebal=sorted({di[d]+1 for tk in tickers for d in sig[tk].index if d in di and di[d]+1 < N})
    prob_at={i:{} for i in rebal}; elig_at={i:{} for i in rebal}
    for tk in tickers:
        p=sig[tk]["prob"].to_numpy(); e=sig[tk]["elig"].to_numpy()
        for j,d in enumerate(sig[tk].index):
            if d in di and di[d]+1 < N: i=di[d]+1; prob_at[i][tk]=p[j]; elig_at[i][tk]=bool(e[j])
    return dict(days=days,O=O,H=H,L=L,C=C,rebal=set(rebal),prob_at=prob_at,elig_at=elig_at)

In [ ]:
# The fast backtest (grid engine) + a one-line helper
# EW-Selected + TP/SL: at each rebalance buy top-K eligible names by prob (free cash / new names, at the open);
# daily TP/SL on High/Low with SL winning ties; hold otherwise, sell at a rebalance when the signal dies.
# Equity is marked DAILY, so the max drawdown is comparable to the 1/N benchmark. Returns (end_capital, mdd%).
def tpsl_run(P, K, tp=0.25, sl=0.15):
    O,H,L,C,rebal,prob_at,elig_at = P["O"],P["H"],P["L"],P["C"],P["rebal"],P["prob_at"],P["elig_at"]
    cash=CAPITAL; pos={}; V=[CAPITAL]
    for i in range(len(P["days"])):
        for tk in list(pos):
            lo=L[tk][i]
            if lo!=lo: continue
            sh,e=pos[tk]
            if   lo<=e*(1-sl): fill=e*(1-sl)
            elif H[tk][i]>=e*(1+tp): fill=e*(1+tp)
            else: continue
            cash+=sh*fill*(1-FEE); del pos[tk]
        if i in rebal:
            ea,pa=elig_at[i],prob_at[i]
            for tk in list(pos):
                if tk in ea and not ea[tk]: cash+=pos[tk][0]*O[tk][i]*(1-FEE); del pos[tk]
            slots=K-len(pos)
            if slots>0:
                cand=[tk for tk in ea if ea[tk] and tk not in pos]; cand.sort(key=lambda tk:-pa[tk])
                buy=cand[:slots]
                if buy:
                    each=cash/len(buy)
                    for tk in buy:
                        px=O[tk][i]; pos[tk]=((each/(1+FEE))/px, px); cash-=each
        V.append(cash + sum(sh*C[tk][i] for tk,(sh,e) in pos.items() if C[tk][i]==C[tk][i]))
    end=V[-1]; V=np.array(V); mdd=float((1 - V/np.maximum.accumulate(V)).max()*100)
    return round(end), round(mdd,2)

def ew_tpsl_fast(df, cfg, use_regime=False):
    hi_map=fit_hi(df_train,cfg); trad=build_trad_cfg(df_train,cfg)      # thresholds always from TRAIN
    return tpsl_run(tpsl_prep(df,cfg,hi_map,trad,use_regime), cfg["k"])

In [ ]:
# The logging engine for ONE config - full trade log (tx) + daily equity (eq)
# Same rules as tpsl_run (next-bar execution, daily equity so drawdown matches 1/N). Pass tp=0.25, sl=0.15
# to reconcile with the grid to the dollar. Prints end / PnL / realized / fees and the strategy-vs-1/N drawdown.
def ew_tpsl(df, cfg, use_regime=False, tp=0.25, sl=0.15):
    K = cfg["k"]
    hi_map = fit_hi(df_train, cfg); trad = build_trad_cfg(df_train, cfg)
    tickers, sig, daily = tpsl_data(df, cfg, hi_map, trad, use_regime)
    all_days = sorted(set().union(*[set(daily[tk].index) for tk in tickers]))
    pos_of = {d:i for i,d in enumerate(all_days)}
    exec_sig = {}
    for tk in tickers:
        for a in sig[tk].index:
            if a in pos_of and pos_of[a]+1 < len(all_days):
                ed = all_days[pos_of[a]+1]
                exec_sig.setdefault(ed, {})[tk] = (float(sig[tk].loc[a,"prob"]), bool(sig[tk].loc[a,"elig"]))
    rebal = set(exec_sig)
    cash = CAPITAL; pos = {}; tx = []; eq = []
    for t in all_days:
        for tk in list(pos):
            if t not in daily[tk].index: continue
            bar = daily[tk].loc[t]; e = pos[tk]["entry"]; sh = pos[tk]["shares"]
            hit = fill = None
            if bar["Low"]  <= e*(1-sl): hit, fill = "SL", e*(1-sl)
            elif bar["High"] >= e*(1+tp): hit, fill = "TP", e*(1+tp)
            if hit:
                gross = sh*fill; fee = FEE*gross; cash += gross-fee
                tx.append({"Date":t.date(),"ticker":tk,"side":"SELL","reason":hit,"shares":round(sh,3),
                           "price":round(fill,2),"fee":round(fee,2),"pnl":round(sh*(fill-e)-fee,2)}); del pos[tk]
        if t in rebal:
            sigt = exec_sig[t]
            for tk in list(pos):
                if tk in sigt and not sigt[tk][1] and t in daily[tk].index:
                    px = float(daily[tk].loc[t,"Open"]); sh = pos[tk]["shares"]; e = pos[tk]["entry"]
                    gross = sh*px; fee = FEE*gross; cash += gross-fee
                    tx.append({"Date":t.date(),"ticker":tk,"side":"SELL","reason":"signal","shares":round(sh,3),
                               "price":round(px,2),"fee":round(fee,2),"pnl":round(sh*(px-e)-fee,2)}); del pos[tk]
            slots = K - len(pos)
            if slots > 0:
                cand = [tk for tk in sigt if sigt[tk][1] and tk not in pos and t in daily[tk].index]
                cand.sort(key=lambda tk: -sigt[tk][0]); buy = cand[:slots]
                if buy:
                    each = cash/len(buy)
                    for tk in buy:
                        px = float(daily[tk].loc[t,"Open"]); sh = (each/(1+FEE))/px
                        gross = sh*px; fee = FEE*gross; cash -= gross+fee; pos[tk] = {"shares":sh,"entry":px}
                        tx.append({"Date":t.date(),"ticker":tk,"side":"BUY","reason":"entry","shares":round(sh,3),
                                   "price":round(px,2),"fee":round(fee,2),"pnl":np.nan})
        val = cash + sum(pos[tk]["shares"]*float(daily[tk].loc[t,"Close"]) for tk in pos if t in daily[tk].index)
        eq.append({"Date":t.date(),"value":round(val,0),"cash":round(cash,0),"n_held":len(pos),"held":", ".join(pos)})
    end_val = cash + sum(pos[tk]["shares"]*float(daily[tk]["Close"].iloc[-1]) for tk in pos)
    unreal  = sum(pos[tk]["shares"]*(float(daily[tk]["Close"].iloc[-1])-pos[tk]["entry"]) for tk in pos)
    tx = pd.DataFrame(tx); eq = pd.DataFrame(eq)
    eq["pnl_$"]      = eq["value"].diff().round(0)
    eq["cum_ret_%"]  = ((eq["value"]/CAPITAL-1)*100).round(2)
    eq["drawdown_%"] = ((eq["value"]/eq["value"].cummax()-1)*100).round(2)
    realized = tx[tx.side=="SELL"].pnl.sum() if len(tx) else 0.0
    strat_mdd = (1 - eq["value"]/eq["value"].cummax()).max()*100 if len(eq) else 0.0
    onen_s = portfolio_1overN(df); onen = onen_s.iloc[-1]; onen_mdd = mdd_of(onen_s)
    scen = "val" if df is df_validation else "train"
    print(f"EW+TPSL K={K} | {scen} | GMM={use_regime}: end {end_val:,.0f} | PnL {end_val-CAPITAL:,.0f} | "
          f"realized {realized:,.0f} | unrealized {unreal:,.0f} | fees {tx.fee.sum() if len(tx) else 0:,.0f} | trades {len(tx)}")
    print(f"1/N   {scen}: end {onen:,.0f} | PnL {onen-CAPITAL:,.0f}   ->   EW+TPSL "
          f"{'beats' if end_val>onen else 'loses to'} 1/N by {end_val-onen:,.0f}")
    print(f"max drawdown: EW+TPSL {strat_mdd:.1f}%  |  1/N {onen_mdd:.1f}%")
    return tx, eq

In [ ]:
# significance test statistics + helpers (run the backtester cells above first)
import io, contextlib, os, numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy import stats

RF_PP = 0.02/252
GRID  = pd.read_parquet(f"{DATA_DIR}/Backtest_results/results_all.parquet")
os.makedirs(f"{DATA_DIR}/Article_figures", exist_ok=True)
fusion_for = lambda sc: "single" if sc == "daily" else "together"
pfmt = lambda p: "$<$0.001" if p < 0.001 else f"{p:.3f}"

def save_fig(fig, name):
    fig.savefig(f"{DATA_DIR}/Article_figures/{name}.png", dpi=200, bbox_inches="tight")
    fig.savefig(f"{DATA_DIR}/Article_figures/{name}.pdf", bbox_inches="tight")

def pesaran_timmermann(pred_up, act_up):
    x, y = np.asarray(pred_up, float), np.asarray(act_up, float); n = len(y)
    P = np.mean(x == y); Py, Px = y.mean(), x.mean(); Ps = Py*Px + (1-Py)*(1-Px)
    v = Ps*(1-Ps)/n - ((2*Py-1)**2*Px*(1-Px) + (2*Px-1)**2*Py*(1-Py) + 4*Py*Px*(1-Py)*(1-Px)/n)/n
    if v <= 0: return np.nan, np.nan
    s = (P-Ps)/np.sqrt(v); return s, stats.norm.sf(s)

def dm_hln(l1, l2, h=1):
    d = np.asarray(l1)-np.asarray(l2); n = len(d); dbar = d.mean()
    dm = dbar/np.sqrt(np.mean((d-dbar)**2)/n) * np.sqrt((n+1-2*h+h*(h-1)/n)/n)
    return dm, 2*stats.t.sf(abs(dm), df=n-1)

def sharpe(r): e = np.asarray(r)-RF_PP; return e.mean()/e.std(ddof=1)

def jk_memmel(ra, rb):
    ra, rb = np.asarray(ra), np.asarray(rb); n = len(ra)
    sa, sb = sharpe(ra), sharpe(rb); rho = np.corrcoef(ra, rb)[0,1]
    th = (1/n)*(2 - 2*rho + 0.5*(sa**2 + sb**2 - 2*sa*sb*rho**2))
    return sa, sb, (sa-sb)/np.sqrt(th), 2*stats.norm.sf(abs((sa-sb)/np.sqrt(th)))

def anatolyev_gerko(expo, mkt):
    c = (np.asarray(expo)-np.mean(expo))*(np.asarray(mkt)-np.mean(mkt))
    z = c.mean()/(c.std(ddof=1)/np.sqrt(len(c))); return c.mean(), z, 2*stats.norm.sf(abs(z))

def deflated_sharpe(sr, T, sk, ku, var_trials, n_trials):
    g = 0.5772156649
    sr0 = np.sqrt(var_trials)*((1-g)*stats.norm.ppf(1-1/n_trials) + g*stats.norm.ppf(1-1/(n_trials*np.e)))
    z = (sr-sr0)*np.sqrt(T-1)/np.sqrt(1 - sk*sr + (ku-1)/4*sr**2)
    return stats.norm.cdf(z), sr0

def cfg_of(row):
    c = {k: row[k] for k in ["algo","tier","scale","fusion","train_start"]}
    c["x_days"], c["k"] = int(row["x_days"]), int(row["k"]); return c

def strat_value(cfg, use_regime):
    with contextlib.redirect_stdout(io.StringIO()):
        _, eq = ew_tpsl(df_validation, cfg, use_regime=use_regime)
    idx = pd.to_datetime(eq["Date"])
    return pd.Series(eq["value"].values, index=idx), pd.Series(eq["cash"].values, index=idx)

In [ ]:
# compute the battery, emit paper-ready LaTeX (Tables 7-8) + fig_significance
MAIN_START, MAIN_X, MAIN_TIER, SAMPLE = "2015-01-01", 10, "gmm", 150

rows = []
for algo in ["lstm","xgb","knn"]:
    for scale in ["daily","weekly","monthly"]:
        d = df_validation[(df_validation.algo==algo)&(df_validation.scale==scale)&(df_validation.tier==MAIN_TIER)
                          &(df_validation.train_start==MAIN_START)&(df_validation.x_days==MAIN_X)
                          &(df_validation.fusion==fusion_for(scale))]
        if d.empty: continue
        y, p = d.y_up.to_numpy(), d.prob_up.to_numpy()
        pt_s, pt_p = pesaran_timmermann(p>0.5, y)
        dm_s, dm_p = dm_hln((p-y)**2, (y.mean()-y)**2)
        rows.append([algo.upper(), scale.title(), len(d), np.mean((p>0.5)==y), pt_s, pt_p, dm_s, dm_p])
D = pd.DataFrame(rows, columns=["algo","scale","n","acc","PTz","PTp","DMt","DMp"])

best = GRID.loc[GRID["end_capital_train"].idxmax()]; cfg = cfg_of(best)
gate = str(best["gmm"]) in ("True","true","1")
sv, cash = strat_value(cfg, gate)
onen = portfolio_1overN(df_validation); onen.index = pd.to_datetime(onen.index)
i = sv.index.intersection(onen.index)
rs, rb = sv.reindex(i).pct_change().dropna(), onen.reindex(i).pct_change().dropna()
i = rs.index.intersection(rb.index); rs, rb = rs[i], rb[i]
sa, sb, jkz, jkp = jk_memmel(rs.values, rb.values)
expo = ((sv-cash)/sv).reindex(i).shift(1).dropna(); j = expo.index.intersection(rb.index)
agEP, agz, agp = anatolyev_gerko(expo[j].values, rb[j].values)

srs, best_r = [], None
for _, r in GRID.sample(SAMPLE, random_state=0).iterrows():
    try:
        v, _ = strat_value(cfg_of(r), str(r["gmm"]) in ("True","true","1"))
        ri = v.pct_change().dropna().values
        if len(ri) < 30 or ri.std() == 0: continue
        s = sharpe(ri); srs.append(s)
        if best_r is None or s > sharpe(best_r): best_r = ri
    except Exception: continue
srs = np.array(srs); srb = sharpe(best_r)
dsr, sr0 = deflated_sharpe(srb, len(best_r), stats.skew(best_r),
                           stats.kurtosis(best_r, fisher=False), srs.var(ddof=1), len(GRID))
ann = np.sqrt(252)

sig = lambda p: "significant" if p < 0.05 else "not significant"
print(r"\begin{table}[!ht]\centering\caption{Significance tests on the held-out test period. "
      r"No test finds exploitable skill.}\label{tab:sig}")
print(r"\footnotesize\setlength{\tabcolsep}{4pt}")
print(r"\begin{tabular}{lll}\toprule")
print(r"Test & Statistic & Result \\ \midrule")
print(r"Directional skill (Pesaran--Timmermann) & median $z=%.2f$ & no skill (%d/%d sig.) \\"
      % (D.PTz.median(), (D.PTp<0.05).sum(), len(D)))
print(r"Probabilistic accuracy (Diebold--Mariano) & median $t=%.2f$ & no gain over base rate ($%d/%d$ worse) \\"
      % (D.DMt.median(), int(((D.DMt>0)&(D.DMp<0.05)).sum()), len(D)))
print(r"Sharpe vs.\ 1/N (Jobson--Korkie/Memmel) & $z=%.2f$, $p=%s$ & %s \\"
      % (jkz, pfmt(jkp), "no difference" if jkp>=0.05 else "different"))
print(r"Market timing (Anatolyev--Gerko) & $z=%.2f$, $p=%s$ & %s \\"
      % (agz, pfmt(agp), "no timing" if agp>=0.05 else "timing"))
print(r"Best Sharpe deflated ($N=%d$ trials) & DSR $=%.3f$ & %s \\"
      % (len(GRID), dsr, sig(1-dsr) if dsr>0.95 else "not significant"))
print(r"\bottomrule\end{tabular}\end{table}")

print(r"\begin{table*}[tb]\centering\caption{Directional and accuracy tests per predictor and horizon "
      r"(test period; per-ticker GMM, 2015 start, $T=10$).}\label{tab:sigdetail}")
print(r"\footnotesize")
print(r"\begin{tabular}{llrrrrrr}\toprule")
print(r"Model & Horizon & $n$ & Accuracy & PT $z$ & PT $p$ & DM $t$ & DM $p$ \\ \midrule")
for _, r in D.iterrows():
    print(r"%s & %s & %d & %.3f & %.2f & %s & %.2f & %s \\"
          % (r.algo, r.scale, r.n, r.acc, r.PTz, pfmt(r.PTp), r.DMt, pfmt(r.DMp)))
print(r"\bottomrule\end{tabular}\end{table*}")

fig, a = plt.subplots(figsize=(7, 4.2))
a.hist(srs*ann, bins=30, color="#c9c9c9", edgecolor="white")
a.axvline(srb*ann, color="#2c6fbb", lw=2, label=f"best strategy ({srb*ann:.2f})")
a.axvline(sr0*ann, color="#c0392b", ls="--", lw=2, label=f"deflated threshold ({sr0*ann:.2f})")
a.axvline(sb*ann,  color="black",  ls=":",  lw=2, label=f"1/N ({sb*ann:.2f})")
a.set_xlabel("Annualised Sharpe ratio"); a.set_ylabel("Configurations")
a.set_title("Sharpe ratios across the configuration search", color="#555"); a.legend(fontsize=8)
fig.tight_layout(); save_fig(fig, "fig_significance"); plt.show()

print(r"""
% ---- figure block ----
\begin{figure}[!ht]\centering\includegraphics[width=\linewidth]{figures/fig_significance.png}
\caption{Annualised Sharpe ratios across the configuration search. The best strategy does not clear the
deflated-Sharpe threshold implied by the size of the search, and 1/N sits among the strategies.}\label{fig:sig}\end{figure}""")